# Gait Analysis Exploration Notebook

This notebook provides an interactive environment for exploring the gait analysis system.


In [ ]:
# Setup - add src to path for development
import sys
sys.path.insert(0, '../src')

import cv2
import numpy as np
from pathlib import Path
import json

from gait_analysis import KeypointDetector, GaitAnalyzer, GaitVisualizer
from gait_analysis.main import analyze_video


## 1. Load and Test Keypoint Detection

First, let's test the YOLO pose estimation on a single frame.


In [ ]:
# Initialize detector (downloads model if needed)
detector = KeypointDetector(model_name="yolov8n-pose.pt")
print("Detector initialized successfully!")


In [ ]:
# Replace with your video path
VIDEO_PATH = "../data/sample_walking.mp4"

# Check if video exists
if Path(VIDEO_PATH).exists():
    print(f"Video found: {VIDEO_PATH}")
else:
    print(f"Video not found: {VIDEO_PATH}")
    print("Please place a walking video in the data/ folder and update VIDEO_PATH")


In [ ]:
# Extract and analyze a single frame
if Path(VIDEO_PATH).exists():
    cap = cv2.VideoCapture(VIDEO_PATH)
    
    # Skip to middle of video
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, total_frames // 2)
    
    ret, frame = cap.read()
    cap.release()
    
    if ret:
        # Detect keypoints
        keypoints = detector.detect_frame(frame, 0, 0.0)
        
        if keypoints:
            print("Detected keypoints:")
            for name, (x, y, conf) in keypoints.keypoints.items():
                print(f"  {name}: ({x:.1f}, {y:.1f}) conf={conf:.2f}")
            
            # Get YOLO visualization
            results = detector.get_results_with_visualization(frame)
            annotated = results[0].plot()
            
            # Display (convert BGR to RGB for matplotlib)
            import matplotlib.pyplot as plt
            plt.figure(figsize=(12, 8))
            plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
            plt.title("Detected Keypoints")
            plt.axis('off')
            plt.show()
        else:
            print("No person detected in frame")


## 2. Run Full Analysis

Analyze the complete video and examine results.


In [ ]:
# Run analysis (without visualization for notebooks)
if Path(VIDEO_PATH).exists():
    results = analyze_video(
        video_path=VIDEO_PATH,
        output_path="../data/analysis_results.json",
        show_visualization=False  # Set to True to show window
    )
    print("\nAnalysis complete!")


In [ ]:
# Display results
if 'results' in dir():
    print(json.dumps(results, indent=2))


In [ ]:
import matplotlib.pyplot as plt

if 'results' in dir() and results['summary']['total_steps'] > 0:
    # Create visualization
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Pie chart of step types
    labels = ['Correct (Heel Strike)', 'Incorrect (Toe Strike)']
    sizes = [results['summary']['correct_steps'], results['summary']['incorrect_steps']]
    colors = ['#4CAF50', '#f44336']
    
    axes[0].pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
    axes[0].set_title('Step Classification Distribution')
    
    # Bar chart per foot
    feet = ['Left', 'Right']
    correct = [results['per_foot']['left']['correct_steps'], 
               results['per_foot']['right']['correct_steps']]
    incorrect = [results['per_foot']['left']['incorrect_steps'],
                 results['per_foot']['right']['incorrect_steps']]
    
    x = np.arange(len(feet))
    width = 0.35
    
    axes[1].bar(x - width/2, correct, width, label='Correct', color='#4CAF50')
    axes[1].bar(x + width/2, incorrect, width, label='Incorrect', color='#f44336')
    axes[1].set_xlabel('Foot')
    axes[1].set_ylabel('Number of Steps')
    axes[1].set_title('Steps per Foot')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(feet)
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()
else:
    print("No step data available - run analysis first")


## 4. Step Timing Analysis


In [ ]:
if 'results' in dir() and results['summary']['total_steps'] > 0:
    # Extract step durations
    all_steps = results['detailed_steps']['left'] + results['detailed_steps']['right']
    
    heel_durations = [s['duration_ms'] for s in all_steps if s['step_type'] == 'heel_strike']
    toe_durations = [s['duration_ms'] for s in all_steps if s['step_type'] == 'toe_strike']
    
    fig, ax = plt.subplots(figsize=(10, 5))
    
    if heel_durations:
        ax.hist(heel_durations, bins=10, alpha=0.7, label='Heel Strike', color='#4CAF50')
    if toe_durations:
        ax.hist(toe_durations, bins=10, alpha=0.7, label='Toe Strike', color='#f44336')
    
    ax.set_xlabel('Step Duration (ms)')
    ax.set_ylabel('Frequency')
    ax.set_title('Step Duration Distribution by Type')
    ax.legend()
    
    plt.show()
    
    # Statistics
    print("\nStep Duration Statistics:")
    print(f"  Heel Strike: mean={np.mean(heel_durations):.1f}ms, std={np.std(heel_durations):.1f}ms" if heel_durations else "  No heel strike data")
    print(f"  Toe Strike: mean={np.mean(toe_durations):.1f}ms, std={np.std(toe_durations):.1f}ms" if toe_durations else "  No toe strike data")


## 5. Next Steps

1. **Fine-tune parameters**: Adjust velocity thresholds in `FootTracker` for better step detection
2. **Test with different videos**: Try videos with different camera angles and lighting
3. **Compare models**: Test with different YOLO model sizes for accuracy vs speed tradeoffs
4. **Add custom metrics**: Extend `GaitAnalyzer` with domain-specific gait metrics
